# Top-level comments: exploratory analysis

A top-level comment replies directly to the submission (its `parent_id` starts with `t3_`); every other comment is a reply to another comment (`t1_`). This notebook looks only at top-level comments.

## 1. Setup and loading

In [1]:
import html
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.core.error import StdinNotImplementedError
from IPython.display import clear_output

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    raise FileNotFoundError("Could not find project root with data/ directory.")

CLEAN = ROOT / "data" / "cleaned"
COMMENTS_PATH = CLEAN / "comments_clean.parquet"
SUBMISSIONS_PATH = CLEAN / "submissions_clean.parquet"

PERIODS = ["T1_Y1", "T1_Y2", "T2_Y1", "T2_Y2"]
SUBREDDITS = ["politics", "AskTrumpSupporters", "Conservative", "PoliticalDiscussion", "democrats", "Republican"]

The comments table has about 4.5M rows, so it is read without the text columns `body` and `body_clean`.

In [2]:
comments_file = pq.ParquetFile(COMMENTS_PATH)
TEXT_COLUMNS = ["body", "body_clean"]
light_columns = [c for c in comments_file.schema_arrow.names if c not in TEXT_COLUMNS]
com = pd.read_parquet(COMMENTS_PATH, columns=light_columns)
com.shape

(4514780, 24)

## 2. Selecting top-level comments

In [3]:
# is_top_level is set in reddit_cleaning.ipynb; check it against parent_id before using it
assert (com["is_top_level"] == com["parent_id"].str.startswith("t3_")).all()

top = com[com["is_top_level"]].reset_index(drop=True)

n_all, n_top = len(com), len(top)
print(f"All comments:       {n_all:,}")
print(f"Top-level comments: {n_top:,} ({n_top / n_all:.1%} of all comments)")

All comments:       4,514,780
Top-level comments: 1,397,476 (31.0% of all comments)


## 3. Manual labels

**Goal.** Both of us label the same 500 top-level comments by hand. Comparing the two sets of labels later shows how often two people disagree about the stance of a comment.

### 3.1 Sample

Rules of the draw:

- `"Trump"` is in the comment's own text and not only in a quote, and the comment is between `MIN_WORDS` and `MAX_WORDS` words long.
- The post of the thread is in `submissions_clean.parquet`, so it can be shown next to the comment.
- The same number of comments comes from each of the six subreddits, and inside a subreddit from each of the four windows (the same rule as section 1.1 of `model_selection.ipynb`). Otherwise r/politics would give almost all of the sample.
- The sample is saved to `data/top_comments/label_sample.csv` and read from there on every later run. `data/` is not shared between our computers, so the second annotator copies this file instead of drawing the sample again.

In [4]:
LABEL_DIR = ROOT / "data" / "top_comments"
LABEL_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_PATH = LABEL_DIR / "label_sample.csv"

N_SAMPLE = 500                # comments in the sample; both annotators label all of them
MIN_WORDS, MAX_WORDS = 5, 250 # length limits of a comment, in words
OVERSAMPLE = 60               # candidates drawn per subreddit and window before their text is checked
SEED = 42
REBUILD_SAMPLE = False        # True draws a new sample and overwrites the file; labels of replaced comments no longer apply


def quotas(total, groups, offset=0):
    """Split `total` as evenly as possible over `groups`. The remainder goes to the groups starting at position `offset`."""
    base, extra = divmod(total, len(groups))
    return {g: base + ((i - offset) % len(groups) < extra) for i, g in enumerate(groups)}


def stratified(pool, total):
    """`total` rows of a shuffled `pool`: equal numbers per subreddit, then equal numbers per window inside a subreddit."""
    by_size = list(pool["subreddit"].value_counts().index)     # a remainder goes to the subreddits with the most eligible comments
    picked = []
    for i, (s, n_subreddit) in enumerate(quotas(total, by_size).items()):
        for p, n in quotas(n_subreddit, PERIODS, offset=i).items():
            stratum = pool.loc[(pool["subreddit"] == s) & (pool["period"] == p)].head(n)
            assert len(stratum) == n, f"not enough comments in r/{s}, {p}"
            picked.append(stratum)
    return pd.concat(picked)


def draw_sample():
    # candidates are chosen from the metadata, and the text is read only for them
    pool = top.loc[~top["trump_only_in_quote"], ["id", "submission_id", "subreddit", "period"]]
    candidates = pool.groupby(["subreddit", "period"]).sample(n=OVERSAMPLE, random_state=SEED)
    comments = pq.read_table(COMMENTS_PATH, columns=["id", "body_clean"],
                             filters=[("id", "in", candidates["id"].tolist())]).to_pandas()
    posts = pq.read_table(SUBMISSIONS_PATH, columns=["id", "title", "selftext", "domain"],
                          filters=[("id", "in", candidates["submission_id"].unique().tolist())]).to_pandas()
    posts.columns = ["submission_id", "post_title", "post_body", "post_domain"]

    rows = (candidates
            .merge(comments.rename(columns={"body_clean": "comment"}), on="id", validate="one_to_one")
            .merge(posts, on="submission_id", validate="many_to_one"))     # comments whose post is missing are dropped
    rows = rows.loc[rows["comment"].str.count(r"\S+").between(MIN_WORDS, MAX_WORDS)
                    & rows["comment"].str.contains("trump", case=False)]

    out = stratified(rows.sample(frac=1, random_state=SEED), N_SAMPLE).sample(frac=1, random_state=SEED)
    out["order"] = np.arange(1, len(out) + 1)                  # reading order, the same for both annotators
    assert out["id"].is_unique
    return out[["id", "submission_id", "subreddit", "period", "order", "post_title", "post_domain", "post_body", "comment"]]


if REBUILD_SAMPLE or not SAMPLE_PATH.exists():
    draw_sample().to_csv(SAMPLE_PATH, index=False, lineterminator="\n")
sample = pd.read_csv(SAMPLE_PATH, dtype=str, keep_default_na=False)    # an empty post body stays an empty string
sample["order"] = sample["order"].astype(int)

composition = pd.crosstab(sample["subreddit"], sample["period"]).reindex(index=SUBREDDITS, columns=PERIODS)
composition["all"] = composition.sum(axis=1)
composition.loc["All"] = composition.sum()
print(f"{len(sample)} comments, median {sample['comment'].str.count(r'\S+').median():.0f} words")
composition

500 comments, median 46 words


period,T1_Y1,T1_Y2,T2_Y1,T2_Y2,all
subreddit,,,,,
politics,21,21,20,21,83
AskTrumpSupporters,21,21,21,20,83
Conservative,21,21,21,21,84
PoliticalDiscussion,20,21,21,21,83
democrats,21,20,21,21,83
Republican,21,21,21,21,84
All,125,125,125,125,500


### 3.2 Labelling

Each of us runs the cell with our own number. The cell shows one comment at a time together with the post of its thread and asks for a label. The subreddit, the window, the author's flair and the model's prediction are hidden, so they do not influence the label.

**What to label:** the attitude of the comment's author toward Trump or his decisions, as far as the comment shows it.

| Key | Label | When |
|---|---|---|
| `a` | `against` | The author criticises, mocks or opposes Trump, his decisions or his administration |
| `n` | `neutral` | No clear attitude: a question, a fact, or a comment about something else that only mentions Trump |
| `p` | `pro` | The author supports, defends or praises Trump or his decisions |
| `s` | skip | The comment cannot be judged (not readable, not in English). Skipped comments are left out of the comparison |

The labels are the same three classes as `student_label`, so they can also be compared with the model later.

Rules for the hard cases, so that both of us decide the same way:

- The post is context only. Use it to understand what the comment answers (for example, sarcasm aimed at the headline), but label the attitude of the comment's author, not of the post.
- Sarcasm and irony are labelled by what the author means, not by the words.
- An attack on Trump's opponents or on the media is `pro` only when the defence of Trump is clear. Otherwise it is `neutral`.
- A comment that criticises one decision and supports Trump in general (or the opposite) gets the label of its main point.
- A quote of Trump with no comment from the author is `neutral`.

Controls: `b` goes back one comment, `q` stops. Every answer is saved at once to `data/top_comments/labels_annotator1.csv` or `labels_annotator2.csv`, and running the cell again continues from the first comment without a label.

In [5]:
KEYS = {"a": "against", "n": "neutral", "p": "pro", "s": "skip"}
LINE_WIDTH = 110        # characters per line when a text is shown
MAX_POST_CHARS = 1500   # a longer post body is cut; the comment is always shown in full


def labels_path(annotator):
    return LABEL_DIR / f"labels_annotator{annotator}.csv"


def load_labels(annotator):
    path = labels_path(annotator)
    if not path.exists():
        return {}
    saved = pd.read_csv(path, dtype=str)
    unknown = set(saved["id"]) - set(sample["id"])
    assert not unknown, f"{path.name} has {len(unknown)} comments that are not in the sample; was the sample drawn again?"
    return dict(saved[["id", "label"]].to_numpy())


def save_labels(annotator, done):
    path = labels_path(annotator)
    tmp = path.with_suffix(".tmp")
    pd.DataFrame(list(done.items()), columns=["id", "label"]).to_csv(tmp, index=False, lineterminator="\n")
    tmp.replace(path)                                          # an interrupted save cannot leave a half-written file


def wrap(text):
    text = html.unescape(text)                                 # the raw post text keeps &amp;, &gt; and similar
    return "\n".join(textwrap.fill(line, LINE_WIDTH) for line in text.splitlines())


def show(row, header):
    clear_output(wait=True)
    print(header + "\n")
    print("POST\n")
    print(wrap(row["post_title"]))
    if not row["post_domain"].startswith("self."):
        print(f"(link to {row['post_domain']})")
    if row["post_body"]:
        body = row["post_body"]
        print("\n" + wrap(body[:MAX_POST_CHARS] + (" [...]" if len(body) > MAX_POST_CHARS else "")))
    print("\n" + "-" * LINE_WIDTH + "\nCOMMENT\n")
    print(wrap(row["comment"]))


def annotate(annotator):
    """Show the sample comments one by one and ask for a label. The answers are saved after every comment."""
    rows = sample.sort_values("order").reset_index(drop=True)
    done = load_labels(annotator)
    i = next((k for k, comment_id in enumerate(rows["id"]) if comment_id not in done), len(rows))
    try:
        while i < len(rows):
            row = rows.loc[i]
            show(row, f"Annotator {annotator} | comment {i + 1} of {len(rows)} | {len(done)} labelled")
            current = f" (now: {done[row['id']]}, Enter keeps it)" if row["id"] in done else ""
            answer = input(f"\n[a] against  [n] neutral  [p] pro  [s] skip  [b] back  [q] stop{current} > ").strip().lower()
            if answer == "q":
                break
            if answer == "b":
                i = max(i - 1, 0)
            elif answer in KEYS:
                done[row["id"]] = KEYS[answer]
                save_labels(annotator, done)
                i += 1
            elif answer == "" and current:
                i += 1
    except (StdinNotImplementedError, KeyboardInterrupt):
        pass                                                   # no keyboard input (for example during an export)
    clear_output(wait=True)
    print(f"Annotator {annotator}: {len(done)} of {len(rows)} comments are labelled"
          + ("" if len(done) == len(rows) else "; run this cell to continue"))

**Annotator 1**

In [ ]:
annotate(annotator=1)

**Annotator 2**

In [6]:
annotate(annotator=2)

Annotator 2: 0 of 500 comments are labelled; run this cell to continue
